# RNN/LSTM Image Captioning

Notebook ini menjalankan bagian Alvin: feature extraction Flickr8k, preprocessing caption, training decoder Keras RNN/LSTM pre-inject, scratch parity, inference, dan evaluasi BLEU-4/METEOR.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "ML-KPEZ" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(PROJECT_ROOT)
print(sys.executable)

/
/usr/local/bin/python3


In [2]:
import numpy as np
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2.20.0
[]


## Paths and Toggles

In [3]:
IMAGES_DIR = PROJECT_ROOT / "data/raw/flickr8k/images"
CAPTIONS_PATH = PROJECT_ROOT / "data/raw/flickr8k/captions/captions.txt"
FEATURES_DIR = PROJECT_ROOT / "data/features/captioning"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/captioning"
MODELS_DIR = PROJECT_ROOT / "models/keras/captioning"
EXPERIMENTS_DIR = PROJECT_ROOT / "artifacts/experiments/captioning"
PREDICTIONS_DIR = PROJECT_ROOT / "artifacts/predictions/captioning"

RUN_FEATURE_EXTRACTION = False
RUN_PREPROCESSING = False
RUN_TRAINING = True
RUN_INIT_INJECT_TRAINING = True
RUN_EVALUATION = True
RUN_BEAM_EVALUATION = True
RUN_BATCH_INFERENCE = False

EVAL_LIMIT_IMAGES = None
EVAL_BACKENDS = "keras,scratch"
MAX_CAPTION_LENGTHS = "10,20,38"
EVAL_BATCH_SIZE = 1024
# None means evaluate every .keras model in MODELS_DIR, including RNN and LSTM variants.
EVAL_MODEL_PATTERNS = None
APPEND_EVAL_TO_EXISTING = True
SKIP_EXISTING_PREDICTIONS = True

BEAM_EVAL_LIMIT_IMAGES = 50
BEAM_EVAL_BACKENDS = "keras"
BEAM_MAX_CAPTION_LENGTHS = "20"
BEAM_EVAL_SEARCHES = "greedy,beam"
BEAM_EVAL_BEAM_WIDTHS = "3,5"
BATCH_INFERENCE_IMAGE_IDS = []

print("images", IMAGES_DIR.exists())
print("captions", CAPTIONS_PATH.exists())
print("features", (FEATURES_DIR / "features.npy").exists())
print("processed", (PROCESSED_DIR / "train.npz").exists())
print("eval model patterns", EVAL_MODEL_PATTERNS)
print("beam evaluation", RUN_BEAM_EVALUATION, BEAM_EVAL_SEARCHES, BEAM_EVAL_BEAM_WIDTHS, BEAM_EVAL_LIMIT_IMAGES)


images False
captions False
features False
processed False


## Feature Extraction

In [4]:
from tubes2_ml.captioning.feature_extraction import FeatureExtractionConfig, extract_flickr8k_features

RUN_FEATURE_EXTRACTION = True

if RUN_FEATURE_EXTRACTION:
    result = extract_flickr8k_features(
        FeatureExtractionConfig(
            images_dir=IMAGES_DIR,
            output_dir=FEATURES_DIR,
            encoder_name="inception_v3",
            batch_size=32,
            overwrite=True,
        )
    )
    print(result)
else:
    print("Feature extraction skipped.")

ModuleNotFoundError: No module named 'tubes2_ml'

## Caption Preprocessing

In [ ]:
from tubes2_ml.captioning.preprocessing import CaptionPreprocessingConfig, preprocess_captions

if RUN_PREPROCESSING:
    result = preprocess_captions(
        CaptionPreprocessingConfig(
            captions_path=CAPTIONS_PATH,
            output_dir=PROCESSED_DIR,
            train_size=6000,
            validation_size=1000,
            test_size=1000,
        )
    )
    print(result)
else:
    print("Preprocessing skipped.")

Preprocessing skipped.


## Training Grid: 12 RNN/LSTM Experiments

In [ ]:
from scripts.run_captioning_experiments import configs_from_yaml, generate_experiment_configs, run_grid

base_model_config, training_config, grid = configs_from_yaml(PROJECT_ROOT / "configs/captioning/hparam_grid.yaml")
experiment_configs = generate_experiment_configs(base_model_config, **grid)
print("Total experiments:", len(experiment_configs))
[config.name for config in experiment_configs]

Total experiments: 12


['rnn_layers1_hidden128',
 'rnn_layers1_hidden512',
 'rnn_layers2_hidden128',
 'rnn_layers2_hidden512',
 'rnn_layers3_hidden128',
 'rnn_layers3_hidden512',
 'lstm_layers1_hidden128',
 'lstm_layers1_hidden512',
 'lstm_layers2_hidden128',
 'lstm_layers2_hidden512',
 'lstm_layers3_hidden128',
 'lstm_layers3_hidden512']

In [ ]:
if RUN_TRAINING:
    training_results = run_grid(experiment_configs, training_config, skip_completed=True)
    print(json.dumps(training_results, indent=2, default=str))
else:
    print("Training skipped. Set RUN_TRAINING = True to train remaining captioning models.")

Training skipped. Set RUN_TRAINING = True to train remaining captioning models.


## Init-Inject Bonus Experiments


In [ ]:
init_base_config, init_training_config, init_grid = configs_from_yaml(PROJECT_ROOT / "configs/captioning/init_inject.yaml")
init_experiment_configs = generate_experiment_configs(init_base_config, **init_grid)
print("Total init-inject experiments:", len(init_experiment_configs))

if RUN_INIT_INJECT_TRAINING:
    init_training_results = run_grid(init_experiment_configs, init_training_config, skip_completed=True)
    print(json.dumps(init_training_results, indent=2, default=str))
else:
    print("Init-inject training skipped. Set RUN_INIT_INJECT_TRAINING = True to train remaining bonus models.")


## Scratch Forward Parity Check

In [ ]:
from tubes2_ml.captioning.models import CaptionDecoderConfig, build_preinject_decoder
from tubes2_ml.scratch.models.rnn_captioner import build_scratch_rnn_captioner_from_keras
from tubes2_ml.scratch.models.lstm_captioner import build_scratch_lstm_captioner_from_keras

for kind, builder in [("rnn", build_scratch_rnn_captioner_from_keras), ("lstm", build_scratch_lstm_captioner_from_keras)]:
    keras_model = build_preinject_decoder(
        CaptionDecoderConfig(
            vocab_size=13,
            feature_dim=7,
            max_caption_length=5,
            embed_dim=4,
            hidden_units=6,
            num_recurrent_layers=2,
            decoder_type=kind,
            name=f"test_{kind}",
        )
    )
    scratch_model = builder(keras_model)
    features = np.random.default_rng(42).normal(size=(3, 7)).astype("float32")
    tokens = np.array([[1, 4, 5, 0, 0], [1, 3, 2, 0, 0], [1, 8, 9, 10, 11]], dtype="int32")
    keras_out = keras_model.predict([features, tokens], verbose=0)
    scratch_out = scratch_model.forward(features, tokens)
    print(kind, keras_out.shape, np.max(np.abs(keras_out - scratch_out)))

rnn (3, 5, 13) 2.2351742e-08
lstm (3, 5, 13) 1.4901161e-08


## Evaluation: BLEU-4, METEOR, Execution Time

In [ ]:
from pathlib import Path
import csv
import json
import shutil
import tempfile
from scripts.evaluate_captioning_experiments import main as evaluate_captioning_main, summarize_results


def normalize_model_patterns(patterns):
    ignored = {None, "", "*", "*.keras"}
    if patterns is None:
        return None
    if isinstance(patterns, str):
        return None if patterns in ignored else [patterns]

    normalized = []
    for pattern in patterns:
        if isinstance(pattern, (list, tuple, set)):
            nested = normalize_model_patterns(pattern)
            if nested:
                normalized.extend(nested)
        elif pattern not in ignored:
            normalized.append(pattern)
    return normalized or None


def prepare_model_subset(source_dir: Path, patterns, subset_root: Path) -> Path:
    patterns = normalize_model_patterns(patterns)
    if patterns is None:
        return source_dir

    selected_by_name = {}
    for pattern in patterns:
        for model_path in sorted(source_dir.glob(pattern)):
            selected_by_name[model_path.name] = model_path
    selected = [selected_by_name[name] for name in sorted(selected_by_name)]
    if not selected:
        raise FileNotFoundError(f"No models match {patterns!r} in {source_dir}")

    subset_dir = subset_root / "selected_models"
    subset_dir.mkdir(parents=True, exist_ok=True)
    for old_link in subset_dir.glob("*.keras"):
        old_link.unlink()

    for model_path in selected:
        target = subset_dir / model_path.name
        try:
            target.symlink_to(model_path.resolve())
        except OSError:
            shutil.copy2(model_path, target)

    print("Selected models:")
    for model_path in selected:
        print("-", model_path.name)
    return subset_dir


def evaluation_key(row: dict) -> tuple[str, str, str, str, str]:
    return (
        str(row.get("model", "")),
        str(row.get("backend", "")),
        str(row.get("search", "")),
        str(row.get("beam_width") or ""),
        str(row.get("max_caption_length", "")),
    )


def read_csv_rows(path: Path) -> tuple[list[str], list[dict]]:
    if not path.exists() or path.stat().st_size == 0:
        return [], []
    with path.open("r", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        return list(reader.fieldnames or []), list(reader)


def write_csv_rows(path: Path, fields: list[str], rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fields})


TIME_FIELDS = {"execution_time_seconds", "seconds_per_image"}
NUMERIC_INT_FIELDS = {"beam_width", "max_caption_length", "model_input_length", "num_predictions", "batch_size"}
NUMERIC_FLOAT_FIELDS = {"bleu4", "meteor", "execution_time_seconds", "seconds_per_image"}


def row_has_missing_or_zero_time(row: dict) -> bool:
    return str(row.get("execution_time_seconds", "")) in {"", "None", "0", "0.0"}


def merge_evaluation_rows(existing_rows: list[dict], new_rows: list[dict]) -> list[dict]:
    merged = {evaluation_key(row): row for row in existing_rows}
    for row in new_rows:
        key = evaluation_key(row)
        if key in merged:
            preserved = dict(row)
            old_predictions_path = merged[key].get("predictions_path")
            if old_predictions_path not in {None, ""}:
                preserved["predictions_path"] = old_predictions_path
            if row_has_missing_or_zero_time(row):
                for field in TIME_FIELDS:
                    old_value = merged[key].get(field)
                    if old_value not in {None, "", "0", "0.0"}:
                        preserved[field] = old_value
            merged[key] = preserved
        else:
            merged[key] = row
    return list(merged.values())


def coerce_evaluation_row_types(row: dict) -> dict:
    typed = dict(row)
    for field in NUMERIC_INT_FIELDS:
        value = typed.get(field)
        if value in {None, "", "None"}:
            typed[field] = None if field == "beam_width" else value
        else:
            typed[field] = int(float(value))
    for field in NUMERIC_FLOAT_FIELDS:
        value = typed.get(field)
        if value in {None, "", "None"}:
            typed[field] = None
        else:
            typed[field] = float(value)
    return typed


def coerce_evaluation_rows_types(rows: list[dict]) -> list[dict]:
    return [coerce_evaluation_row_types(row) for row in rows]


def merge_evaluation_outputs(temp_csv: Path, target_csv: Path, target_json: Path, target_summary_json: Path) -> list[dict]:
    existing_fields, existing_rows = read_csv_rows(target_csv)
    new_fields, new_rows = read_csv_rows(temp_csv)
    merged_rows = merge_evaluation_rows(existing_rows, new_rows)

    fields = list(existing_fields)
    for field in new_fields:
        if field not in fields:
            fields.append(field)
    if not fields and merged_rows:
        fields = list(merged_rows[0].keys())

    typed_merged_rows = coerce_evaluation_rows_types(merged_rows)
    write_csv_rows(target_csv, fields, merged_rows)
    target_json.write_text(json.dumps(typed_merged_rows, indent=2), encoding="utf-8")
    target_summary_json.write_text(json.dumps(summarize_results(typed_merged_rows), indent=2), encoding="utf-8")
    print(f"Merged {len(new_rows)} rows into {target_csv}")
    return new_rows


def run_captioning_evaluation_job(
    *,
    output_stem: str,
    searches: str,
    backends: str,
    max_caption_lengths: str,
    limit_images: int | None,
    beam_widths: str = "3",
) -> list[dict]:
    import sys as _sys

    temp_root = Path(tempfile.mkdtemp(prefix="ml-kpez-caption-eval-"))
    selected_models_dir = prepare_model_subset(MODELS_DIR, EVAL_MODEL_PATTERNS, temp_root)
    temp_csv = temp_root / f"{output_stem}_results.csv"
    temp_json = temp_root / f"{output_stem}_results.json"
    temp_summary_json = temp_root / f"{output_stem}_summary.json"
    temp_qualitative_json = temp_root / f"{output_stem}_qualitative_samples.json"

    old_argv = _sys.argv
    argv = [
        "evaluate_captioning_experiments.py",
        "--models-dir", str(selected_models_dir),
        "--processed-dir", str(PROCESSED_DIR),
        "--features-dir", str(FEATURES_DIR),
        "--captions-path", str(CAPTIONS_PATH),
        "--split", "test",
        "--backends", backends,
        "--searches", searches,
        "--beam-widths", beam_widths,
        "--batch-size", str(EVAL_BATCH_SIZE),
        "--max-caption-lengths", max_caption_lengths,
        "--output-csv", str(temp_csv),
        "--output-json", str(temp_json),
        "--summary-json", str(temp_summary_json),
        "--qualitative-json", str(temp_qualitative_json),
        "--predictions-dir", str(PREDICTIONS_DIR),
    ]
    if limit_images is not None:
        argv.extend(["--limit-images", str(limit_images)])
    if SKIP_EXISTING_PREDICTIONS:
        argv.append("--skip-existing-predictions")

    _sys.argv = argv
    try:
        evaluate_captioning_main()
    finally:
        _sys.argv = old_argv

    if not APPEND_EVAL_TO_EXISTING:
        destination_csv = EXPERIMENTS_DIR / f"{output_stem}_results.csv"
        destination_json = EXPERIMENTS_DIR / f"{output_stem}_results.json"
        destination_summary_json = EXPERIMENTS_DIR / f"{output_stem}_summary.json"
        shutil.copy2(temp_csv, destination_csv)
        shutil.copy2(temp_json, destination_json)
        shutil.copy2(temp_summary_json, destination_summary_json)
        _, rows = read_csv_rows(destination_csv)
        return coerce_evaluation_rows_types(rows)

    return merge_evaluation_outputs(
        temp_csv=temp_csv,
        target_csv=EXPERIMENTS_DIR / f"{output_stem}_results.csv",
        target_json=EXPERIMENTS_DIR / f"{output_stem}_results.json",
        target_summary_json=EXPERIMENTS_DIR / f"{output_stem}_summary.json",
    )


if RUN_EVALUATION:
    normal_rows = run_captioning_evaluation_job(
        output_stem="evaluation",
        searches="greedy",
        backends=EVAL_BACKENDS,
        max_caption_lengths=MAX_CAPTION_LENGTHS,
        limit_images=EVAL_LIMIT_IMAGES,
    )
else:
    print("Evaluation skipped. Set RUN_EVALUATION = True after models are trained.")

if RUN_BEAM_EVALUATION:
    beam_rows = run_captioning_evaluation_job(
        output_stem="beam_evaluation",
        searches=BEAM_EVAL_SEARCHES,
        backends=BEAM_EVAL_BACKENDS,
        max_caption_lengths=BEAM_MAX_CAPTION_LENGTHS,
        limit_images=BEAM_EVAL_LIMIT_IMAGES,
        beam_widths=BEAM_EVAL_BEAM_WIDTHS,
    )
else:
    print("Beam evaluation skipped. Set RUN_BEAM_EVALUATION = True to append beam results.")


Evaluation skipped. Set RUN_EVALUATION = True after models are trained.


## Results Summary

In [ ]:
import csv

summary_path = EXPERIMENTS_DIR / "evaluation_summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), indent=2))

results_path = EXPERIMENTS_DIR / "evaluation_results.csv"
if results_path.exists():
    with results_path.open("r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    rows_sorted = sorted(rows, key=lambda row: float(row["bleu4"]), reverse=True)
    for row in rows_sorted[:10]:
        print(row)
else:
    print("No evaluation results yet.")

No evaluation results yet.


## Qualitative Samples

In [ ]:
qualitative_path = PREDICTIONS_DIR / "qualitative_samples.json"
if qualitative_path.exists():
    payload = json.loads(qualitative_path.read_text(encoding="utf-8"))
    samples = payload.get("samples", [])
    print("source:", payload.get("source_result", {}))
elif (prediction_files := sorted(PREDICTIONS_DIR.glob("*.json"))):
    samples = json.loads(prediction_files[0].read_text(encoding="utf-8"))[:10]
    print(prediction_files[0])
else:
    samples = []
    print("No prediction files yet.")

for sample in samples[:10]:
    print("image_id:", sample["image_id"])
    print("prediction:", sample["caption"])
    print("references:", sample.get("references", [])[:2])
    print()


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 8)

## Batch Caption Inference


In [ ]:
from tubes2_ml.captioning.inference import generate_captions

if RUN_BATCH_INFERENCE:
    candidate_models = sorted(MODELS_DIR.glob("*.keras"))
    if not candidate_models:
        raise FileNotFoundError(f"No trained models found in {MODELS_DIR}")
    image_ids = BATCH_INFERENCE_IMAGE_IDS
    if not image_ids:
        split = np.load(PROCESSED_DIR / "test.npz")
        image_ids = list(dict.fromkeys(str(image_id) for image_id in split["image_ids"]))[:5]
    batch_predictions = generate_captions(
        model_path=candidate_models[0],
        vocabulary_path=PROCESSED_DIR / "vocabulary.json",
        features_dir=FEATURES_DIR,
        image_ids=image_ids,
        max_caption_length=38,
        backend="scratch",
        search="greedy",
    )
    print(json.dumps(batch_predictions, indent=2))
else:
    print("Batch inference skipped. Set RUN_BATCH_INFERENCE = True after at least one pre-inject model is trained.")
